# Pipeline step 2 (post_train_2): refusal direction + stable target layers

**Runs on a Colab GPU (G4, T4 fallback)**, loading the real `CohereLabs/tiny-aya-water`
HF checkpoint in bf16.

**Known limitation (unchanged):** full CircuitKIT attention-head discovery needs
TransformerLens, which does not support Tiny Aya's architecture. This uses SafeTune's
model-agnostic tools instead: the per-layer refusal direction (Arditi et al.:
mean last-token activation on harmful minus harmless prompts) and
`identify_safety_neurons` (MLP `down_proj` neurons most aligned with that direction).

**Fixes vs. earlier runs:**
1. **48-60 prompts per condition per language** (every eligible `tune`-split prompt;
   pipeline 2 helps build the fix, so it never touches validation's `analysis` seeds) instead of 8. At n=8, three independent
   samples picked three different target layers ([18,23], [13,18], [25,32]).
2. **Target layers chosen by bootstrap stability**: 30 resamples of the prompts; per
   language, keep the layer that lands in the top-5 most often. Pipeline 4 restricts
   LoRA to these layers.
3. **Layer-index alignment verified**: SafeTune hooks decoder-layer *outputs*
   (`hidden_states[k+1]` in HF terms) and tokenizes with `add_special_tokens=False`; the
   old projection helper used `hidden_states[k]` (one layer early) and default special
   tokens. The helper now matches SafeTune and the alignment is checked numerically.
4. **Held-out projection test**: the direction used for the `us`-vs-`neutral` test is fit
   on half of the neutral prompts and tested on the other half, so the neutral mean is
   no longer measured on the same prompts the direction was fit on.

`layer_idx` (the layer used for the projection test) is SafeTune's middle-layer
fallback (`select_directions=False`; the full validation sweep OOM'd on a T4), so it is
a fixed choice, not a data-driven one -- stated as such.


> **Migrated from the `ez-penta` research repo (experiment `post_train_2`).** The executed original with its outputs from the 2026-11-12 run is [`executed/pipeline2_circuit_discovery.ipynb`](executed/pipeline2_circuit_discovery.ipynb). This copy only changes where files are read and written (see `configs/experiment.json`).

In [ ]:
# Project setup (Colab or local): fetch this repository's code and small data files, then load
# the shared config. Large files come from the project's Hugging Face datasets on first use.
import subprocess
import sys
from pathlib import Path

if Path("/content").exists():                     # Google Colab
    REPO = Path("/content/tiny-aya-safety")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ishane0620/tiny-aya-safety", str(REPO)], check=True)
else:                                             # local checkout: this notebook is in <repo>/notebooks
    REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from tiny_aya_safety.config import CONFIG, adapter_source, artifact, data_path, output_dir

OUT = output_dir()   # everything this notebook writes goes here (override with TAS_OUTPUT_DIR)
print("repo:", REPO, "| outputs:", OUT)

## 0. Setup

In [ ]:
import json
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetune", "transformers", "accelerate", "huggingface_hub"], check=True)

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
import safetune.steer as steer
from safetune.instrumentation.interpret import identify_safety_neurons, SafetyNeuronConfig
import scipy.stats as stats

# Tiny Aya is a gated HF checkpoint; HF_TOKEN is passed in via `colab exec --env`.
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("WARNING: no HF_TOKEN env var found; gated model load will likely fail.")

MODEL_ID = "CohereLabs/tiny-aya-water"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
if device != "cuda":
    print("WARNING: no GPU detected — this will be slow and/or may not reflect the intended T4 run.")


## 1. Load the real Tiny Aya HF checkpoint (bfloat16)

In [ ]:

print("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map=device)
model.eval()
print("Model loaded:", MODEL_ID)


## 2. Load the prompt sample (48-60 per language x condition, `framing=control`, `tune` split only)

In [ ]:

with open(data_path("pipeline2_prompts_v2.json"), encoding="utf-8") as f:
    data = json.load(f)
for lang in ["en", "ja"]:
    print(lang, {k: v for k, v in data[lang].items() if k.startswith("n_")})


## 3. Last-token decoder-layer outputs, captured exactly like SafeTune's extractor

Forward hooks on each decoder layer (SafeTune's `_get_decoder_layers`), last position with
left padding, chat template with `add_generation_prompt=True` and `add_special_tokens=False`.
Runs the base model (`model.model`) so the ~262k-vocabulary logits are never materialized
-- computing those for every position is what ran a T4 out of memory.

In [ ]:

import numpy as np
from safetune._refusal_helpers import _get_decoder_layers

decoder_layers = _get_decoder_layers(model)
n_layers = len(decoder_layers)


def last_token_states(prompts, batch_size=8):
    tokenizer.padding_side = "left"
    captured, per_layer = {}, {}

    def make_hook(k):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            captured[k] = hs[:, -1, :].float().cpu()
        return hook

    handles = [layer.register_forward_hook(make_hook(k)) for k, layer in enumerate(decoder_layers)]
    try:
        for i in range(0, len(prompts), batch_size):
            texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                                   add_generation_prompt=True) for p in prompts[i:i + batch_size]]
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                            add_special_tokens=False).to(device)
            with torch.no_grad():
                model.model(**enc)
            for k, v in captured.items():
                per_layer.setdefault(k, []).append(v)
            captured.clear()
    finally:
        for h in handles:
            h.remove()
    return {k: torch.cat(v) for k, v in per_layer.items()}


states = {lang: {cond: last_token_states(data[lang][cond]) for cond in ["harmful_neutral", "twin_neutral", "harmful_us"]}
          for lang in ["en", "ja"]}
torch.cuda.empty_cache()
print(f"Cached last-token states for {n_layers} decoder layers")


def diff_in_means(harm, harmless, idx_h=None, idx_n=None):
    out = {}
    for k in harm:
        h = harm[k] if idx_h is None else harm[k][idx_h]
        n = harmless[k] if idx_n is None else harmless[k][idx_n]
        d = h.mean(0) - n.mean(0)
        out[k] = d / d.norm()
    return out


## 4. SafeTune's full-sample refusal directions + safety-neuron ranking, and the alignment check

Uses SafeTune's own `SteeringVectorExtractor` -- the exact code `extract_refusal_direction`
runs -- but with `batch_size=4`, since `extract_refusal_direction` hardcodes 16, which
does not fit a T4 with this vocabulary. `layer_idx` reproduces its middle-layer fallback
(`target[len(target) // 2]`).

In [ ]:

from safetune.core.runtime.inference.vector_extraction import SteeringVectorExtractor, VectorExtractionConfig
neuron_config = SafetyNeuronConfig(method="weight", top_k_per_layer=16)


def top_layers(directions, k=5):
    report = identify_safety_neurons(model, directions, neuron_config)
    ranked = sorted(report.per_layer.items(), key=lambda kv: -max(abs(s) for _, s in kv[1]) if kv[1] else 0)
    return [int(layer) for layer, _ in ranked[:k]], report


results = {}
safetune_dirs = {}
for lang in ["en", "ja"]:
    ve_cfg = VectorExtractionConfig(target_layers=list(range(n_layers)), pool_method="last_token",
                                    normalize=True, batch_size=4)
    extractor = SteeringVectorExtractor(model, tokenizer, ve_cfg)
    # Same argument order SafeTune uses: direction = harmful_mean - harmless_mean.
    all_dirs = extractor.extract(safe_prompts=data[lang]["harmful_neutral"], unsafe_prompts=data[lang]["twin_neutral"])
    layer_idx = list(range(n_layers))[n_layers // 2]
    direction = all_dirs[layer_idx]
    torch.cuda.empty_cache()
    safetune_dirs[lang] = (direction, layer_idx, all_dirs)

    mine = diff_in_means(states[lang]["harmful_neutral"], states[lang]["twin_neutral"])
    st = {k: all_dirs[k].float().cpu() for k in all_dirs}
    cos_sim = lambda a, b: float(torch.nn.functional.cosine_similarity(a, b, dim=0))
    cos = [cos_sim(mine[k], st[k]) for k in sorted(st)]
    best_match = {k: max(mine, key=lambda j: cos_sim(mine[j], st[k])) for k in st}
    shifted = np.mean([cos_sim(mine[k + 1], st[k]) for k in sorted(st)[:-1]])
    print(f"{lang}: alignment check, cosine(our direction, SafeTune direction) per layer: "
          f"min={min(cos):.4f} mean={np.mean(cos):.4f}; shifted by one layer: mean={shifted:.4f}")
    # Early layers carry little harmful-vs-harmless signal, so bf16 rounding under different
    # padding (batch 8 vs 4) moves them most; the real test is that each layer matches itself.
    assert all(best_match[k] == k for k in st), f"layer index mismatch: {best_match}"
    assert np.mean(cos) > 0.99 and min(cos) > 0.95, "directions do not match SafeTune's"

    full_top5, report = top_layers(all_dirs)
    results[lang] = {"layer_idx": int(layer_idx), "full_sample_top5": full_top5,
                     "alignment_min_cosine": min(cos), "alignment_mean_cosine": float(np.mean(cos)),
                     "alignment_shifted_mean_cosine": float(shifted)}
    print(f"{lang}: layer_idx={layer_idx} (middle-layer fallback), full-sample top-5 layers={full_top5}")


## 5. Bootstrap stability of the top safety layers (30 resamples)

Each resample draws the harmful and harmless prompts with replacement, recomputes the
per-layer directions from the cached states (verified identical to SafeTune's above), and
re-ranks layers with `identify_safety_neurons`. The target layer per language is the one
that appears in the top-5 most often.

In [ ]:

from collections import Counter

N_BOOT = 30
rng = np.random.default_rng(42)
for lang in ["en", "ja"]:
    harm, harmless = states[lang]["harmful_neutral"], states[lang]["twin_neutral"]
    n_h, n_n = len(next(iter(harm.values()))), len(next(iter(harmless.values())))
    in_top5, top1 = Counter(), Counter()
    device_dtype = next(iter(safetune_dirs[lang][2].values()))
    for _ in range(N_BOOT):
        dirs = diff_in_means(harm, harmless, rng.integers(0, n_h, n_h), rng.integers(0, n_n, n_n))
        dirs = {k: v.to(device_dtype.device, device_dtype.dtype) for k, v in dirs.items()}
        ranked, _ = top_layers(dirs)
        in_top5.update(ranked)
        top1[ranked[0]] += 1
    freq = {int(k): v / N_BOOT for k, v in in_top5.most_common()}
    stable = max(freq, key=lambda k: (freq[k], top1.get(k, 0)))
    results[lang].update({"bootstrap_top5_frequency": freq,
                          "bootstrap_top1_counts": {int(k): v for k, v in top1.most_common()},
                          "stable_layer": int(stable), "stable_layer_top5_rate": freq[stable]})
    print(f"{lang}: top-5 frequency over {N_BOOT} resamples: {dict(list(freq.items())[:8])}")
    print(f"{lang}: most stable layer = {stable} (in top-5 in {freq[stable]:.0%} of resamples)")

target_layers = sorted({results[lang]["stable_layer"] for lang in ["en", "ja"]})
results["target_layers"] = target_layers
print(f"\nTarget layers for pipeline 4's circuit-restricted LoRA: {target_layers}")


## 6. Held-out projection test: does `culture=us` suppress activation along the refusal direction?

Direction fit on the first half of the harmful-neutral and twin-neutral prompts at
`layer_idx`; tested on the held-out second half of harmful-neutral vs all harmful-us prompts.
Same-language and cross-language (direction from one language applied to the other).

In [ ]:

results["projection_tests"] = []
def first_half(lang, cond):
    return len(states[lang][cond][0]) // 2


for direction_lang in ["en", "ja"]:
    layer_idx = results[direction_lang]["layer_idx"]
    h = states[direction_lang]["harmful_neutral"][layer_idx][:first_half(direction_lang, "harmful_neutral")]
    n = states[direction_lang]["twin_neutral"][layer_idx][:first_half(direction_lang, "twin_neutral")]
    direction = h.mean(0) - n.mean(0)
    direction = direction / direction.norm()
    for test_lang in ["en", "ja"]:
        neutral_proj = (states[test_lang]["harmful_neutral"][layer_idx][first_half(test_lang, "harmful_neutral"):] @ direction).numpy()
        us_proj = (states[test_lang]["harmful_us"][layer_idx] @ direction).numpy()
        t_stat, p_val = stats.ttest_ind(neutral_proj, us_proj, equal_var=False)
        gap = float(neutral_proj.mean() - us_proj.mean())
        kind = "same-language" if direction_lang == test_lang else "CROSS-language"
        print(f"direction {direction_lang} (layer {layer_idx}) -> {test_lang} [{kind}]: "
              f"neutral(held-out)={neutral_proj.mean():.3f} us={us_proj.mean():.3f} gap={gap:+.3f} "
              f"Welch t={t_stat:.2f} p={p_val:.4f}")
        results["projection_tests"].append({
            "direction_from": direction_lang, "applied_to": test_lang, "layer_idx": layer_idx,
            "n_neutral_heldout": int(len(neutral_proj)), "n_us": int(len(us_proj)),
            "neutral_mean": float(neutral_proj.mean()), "us_mean": float(us_proj.mean()),
            "gap": gap, "t_stat": float(t_stat), "p_value": float(p_val)})


## 7. Save results for pipeline 4 (`target_layers` is what it uses)

In [ ]:

with open(OUT / "pipeline2_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps({k: v for k, v in results.items() if k != "projection_tests"}, indent=2))
